# Turbine curves and standard presets

Define a turbine once, then evaluate operating points from incoming rotor samples. A scalar speed represents uniform inflow; nonuniform samples use area-weighted cubic rotor-equivalent speed.

Presets: pinned FLORIS 4.6.2 NREL 5 MW, IEA 10 MW and IEA 15 MW power/CT tables. Cp shown here is equivalent electrical Cp. Curve interpolation and the yaw operating approximation are engineering assumptions, not a resolved rotor/controller simulation or farm validation. The accepted wake kernel is unchanged.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import wakemarch
from wakemarch import Turbine, Numerics, available_turbines
print(wakemarch.__version__, wakemarch.__file__)
print('Presets:', available_turbines())
u=np.linspace(3,25,301)
fig,axes=plt.subplots(1,3,figsize=(13,3.5))
for name in available_turbines():
    t=Turbine.preset(name)
    axes[0].plot(u,t.power_at(u)/1e6,label=name)
    axes[1].plot(u,t.cp_at(u))
    axes[2].plot(u,t.ct_at(u))
for ax,y in zip(axes,('Electrical power [MW]','Equivalent electrical Cp','Unyawed CT')):
    ax.set(xlabel='Wind speed [m/s]',ylabel=y);ax.grid(alpha=.2)
axes[0].legend(fontsize=8);axes[2].axhline(1,color='gray',ls=':')
fig.tight_layout();plt.show()

## Supply your own Cp/CT tables

All speeds are m/s, D/H are metres, and power inputs are watts. For aerodynamic Cp, specify generator efficiency. For already electrical Cp use efficiency=1. The conversion below creates power **at table nodes**, then interpolates power linearly. This preserves a rated plateau when its power nodes are equal; it does not interpolate Cp directly between nodes.

The custom table below is illustrative, not a standard turbine. Use documented operating curves for scientific comparisons. Alternatively use `Turbine.from_power_ct(..., power_W=..., ct=...)` when electrical power is already available.

In [ ]:
custom=Turbine.from_cp_ct(
    name='illustrative', D=126, H=90,
    wind_speeds=[0,4,8,12,16,25,26,50],
    cp=[0,.30,.46,.30,.1265625,.0331776,0,0],
    ct=[0,.85,.80,.40,.20,.06,0,0],
    generator_efficiency=.94, rho_ref=1.225,
    yaw_power_exponent=1.88,
)
assert np.isclose(custom.power_at(12),custom.power_at(16))
print('Custom turbine at 8 m/s:',custom.operating_point(8.0))
# Input arrays are copied into immutable tuples; the caller cannot mutate a run's tables.

## Rotor distribution and wake initialization

Choose a preset or `custom`. Unequal-area rotor samples must supply corresponding weights. The example uses two equal-area patches to make the effect explicit; a farm study would sample the whole disk (e.g. 576 equal-area points). CT is evaluated at U_RE with no yaw correction; WakeMarch applies its own accepted thrust-yaw law once.

Power evaluates the curve at U_RE × (rho/rho_ref)^(1/3) × cos(yaw)^(p/3), with no second power multiplier. The table's speed range is the operating range: outside it the turbine is not operating (zero power, no wake). A table CT ≥ 1 (NREL 5 MW near cut-in) is capped at 0.97 for the wake, with a warning; the source curve remains available for inspection. Non-operating turbines return zero power and cannot initialize an active wake. See `docs/TURBINES.md` for the full rules.

In [ ]:
turbine=Turbine.preset('nrel_5MW') # or custom
rotor_velocities=np.array([6.,10.])
weights=np.array([1.,1.])
yaw_deg=20.
point=turbine.operating_point(rotor_velocities,weights=weights,yaw_deg=yaw_deg)
print(point)
assert np.isclose(point.rotor_equivalent_m_s,((6**3+10**3)/2)**(1/3))
model=turbine.wake_model(rotor_velocities,weights=weights,yaw_deg=yaw_deg,
                         TI=.1,numerics=Numerics(h_D=.1))
assert model.CT==point.CT and model.U==point.rotor_equivalent_m_s
result=model.solve(6,progress=True)
x,y=np.meshgrid(np.linspace(0,6*turbine.D,121),np.linspace(-2*turbine.D,2*turbine.D,81))
fig,ax=plt.subplots(figsize=(9,3.5))
im=ax.pcolormesh(x/turbine.D,y/turbine.D,result.sample_streamwise_velocity(x,y,turbine.H)/model.U,shading='auto',vmin=.3,vmax=1)
ax.set(xlabel='x/D',ylabel='y/D',title='Wake initialized at rotor-equivalent operating point')
fig.colorbar(im,ax=ax,label='u/U_RE');fig.tight_layout();plt.show()

The new wake uses a uniform background at U_RE. This factory does not propagate the detailed incoming rotor distribution into downstream wake evolution. The original `WakeModel(D=..., H=..., U=..., CT=..., ...)` interface remains available, including its power-law shear option. This package API does not claim active-farm validation.

## Fixed coefficients for controlled studies

Choose `Turbine.fixed` explicitly when constant CT and CP are appropriate. Its wake uses arithmetic mean incoming speed, while power uses mean cubed speed and a cos³ yaw loss, following the constant-coefficient conventions of the authors' farm study. This is a power approximation without a rated cap or cut-in/out model; it does not reproduce manuscript validation metrics automatically. For a wake alone, use `WakeModel` directly without CP.

In [ ]:
fixed = Turbine.fixed(D=126, H=90, CT=.8, CP=.45)
fixed_point = fixed.operating_point(rotor_velocities, weights=weights, yaw_deg=yaw_deg, rho=1.225)
fixed_model = fixed.wake_model(rotor_velocities, weights=weights, yaw_deg=yaw_deg, TI=.1, numerics=Numerics(h_D=.1))
expected_power = .5 * 1.225 * np.pi * fixed.D**2 / 4 * fixed.CP * np.average(rotor_velocities**3, weights=weights) * np.cos(np.radians(yaw_deg))**3
assert np.isclose(fixed_point.power_W, expected_power)
assert np.isclose(fixed_model.U, np.average(rotor_velocities, weights=weights))
assert fixed_model.CT == fixed.CT
print(f"Fixed mode: wake source {fixed_model.U:.3f} m/s, power {fixed_point.power_W / 1e6:.3f} MW")
print(f"Curve mode: wake source {model.U:.3f} m/s, power {point.power_W / 1e6:.3f} MW")